# 04 · The AI loan officer

**What this notebook does, in plain words**

We give an AI (Gemini) real loan applications and ask how likely each one is to go unpaid. The trick: every application appears **14 times**, identical except for the job title. If the AI's answer moves when only the title changes, the title is what moved it.

It follows `PREREGISTRATION.md` and the decisions recorded after notebook 03.

1. Pick 40 real 2015 applications (10 each from grades A, B, C, D).
2. Make 560 versions: each application × 14 title conditions.
3. Deal the versions into balanced batches of about 25, so no batch holds two versions of the same application and every batch has a similar mix of grades and titles.
4. Send batches to Gemini, save every raw reply, and check every answer.
5. Run everything twice with different batches (`main_1`, `main_2`) to measure how noisy the AI's answers are.

Two pilot runs (`pilot`, `pilot_2`) checked the plumbing first. **They are kept out of all analysis.** This notebook does not analyse job-title effects yet.

## Setup

Loads the libraries, the API key from `.env` (never printed, never committed), and fixes every setting in one place so the run can be repeated exactly.

Every request has a **4-minute time limit**, so a request the server never answers can't hang the run. This is a setting on the connection, not on the model: the model's own settings stay at their defaults.

In [1]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
import httpx2
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(Path(".env"))
REQUEST_TIMEOUT_MS = 240_000                 # 4 minutes per request
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"],
                      http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS))
NETWORK_ERRORS = (httpx.TransportError, httpx2.TransportError)   # dropped connections and timeouts

MODEL = "gemini-3.5-flash-lite"      # default settings: no temperature or other options changed
PROMPT_FILE = Path("prompts/plain.txt")
SEED = 2015                          # one fixed seed for every random choice in this notebook
BATCH_SIZE = 25
PER_GRADE = 10

PREREGISTERED = [
    "software engineer", "police officer", "executive assistant", "attorney", "teacher",
    "registered nurse", "mechanic", "president", "general manager", "owner", "truck driver", "server",
]
CONDITIONS = PREREGISTERED + ["(blank)", "employee"]
print(f"{len(CONDITIONS)} title conditions, model {MODEL}")

14 title conditions, model gemini-3.5-flash-lite


## Step 1 · Pick the 40 base applications

From the 2015 test loans, keep only applications with **every field filled in** and **annual income between \$45,000 and \$90,000**. That income range makes all 14 titles believable: a \$60,000 "attorney" and a \$60,000 "server" are both plausible, so the AI can't dismiss a version as fake.

Then pick 10 at random from each of grades A, B, C and D with the fixed seed. Each gets a short ID such as `A07` (grade A, number 7). That ID is for our analysis only: **the AI never sees it**, because it would reveal the grade.

In [2]:
AI_FIELDS = [
    "emp_years", "annual_inc", "home_ownership", "loan_amnt", "purpose", "fico", "dti", "revol_util",
    "open_acc", "total_acc", "delinq_2yrs", "inq_last_6mths", "pub_rec", "credit_history_years",
]

loans = pd.read_parquet("data/test_2015_scored.parquet")
pool = loans.dropna(subset=AI_FIELDS)
pool = pool[pool["annual_inc"].between(45_000, 90_000) & pool["grade"].isin(list("ABCD"))]
print(f"{len(pool):,} eligible 2015 loans")

base = (
    pd.concat([group.sample(PER_GRADE, random_state=SEED) for _, group in pool.groupby("grade")])
    .sort_values(["grade", "id"])
    .reset_index(drop=True)
)
base.insert(0, "app_id", [f"{g}{i:02d}" for g in "ABCD" for i in range(1, PER_GRADE + 1)])

KEEP = ["app_id", "id", "grade", "sub_grade", "int_rate", "default", "model_risk", "job_title"] + AI_FIELDS
Path("outputs").mkdir(exist_ok=True)
base[KEEP].to_csv("outputs/base_applications.csv", index=False)
print("Saved outputs/base_applications.csv")
base.groupby("grade").agg(apps=("app_id", "size"), defaulted=("default", "sum"),
                          median_income=("annual_inc", "median"), mean_fico=("fico", "mean")).round(0)

132,221 eligible 2015 loans
Saved outputs/base_applications.csv


,apps,defaulted,median_income,mean_fico
grade,,,,
A,10,0,73000.0,728.0
B,10,1,68980.0,686.0
C,10,2,57500.0,674.0
D,10,2,60250.0,690.0


## Step 2 · What the AI sees

Each applicant is written as one line of JSON with plain-English field names. **Only application facts are shown.** The grade, interest rate, state, zip code, name, real outcome and the borrower's real job title are never included.

Two details:
- **Employment length** is written the way LendingClub's form recorded it ("< 1 year", "10+ years"), because the data stores 10 for "10 or more".
- **The blank condition** shows an empty job title (`""`), just as a borrower who left the box empty would look.

In [3]:
def employment_length(years):
    years = int(years)
    if years == 0:
        return "< 1 year"
    if years >= 10:
        return "10+ years"
    return f"{years} year" + ("s" if years > 1 else "")

def applicant_fields(row, title):
    """Everything the AI is allowed to see, in a fixed order. `title` replaces the real job title."""
    return {
        "job title": "" if title == "(blank)" else title,
        "years employed": employment_length(row["emp_years"]),
        "annual income": f"${row['annual_inc']:,.0f}",
        "home ownership": str(row["home_ownership"]).lower(),
        "loan amount": f"${row['loan_amnt']:,.0f}",
        "loan purpose": str(row["purpose"]).replace("_", " "),
        "FICO score": int(round(row["fico"])),
        "debt-to-income ratio": f"{row['dti']:.1f}%",
        "revolving utilization": f"{row['revol_util']:.1f}%",
        "open accounts": int(row["open_acc"]),
        "total accounts": int(row["total_acc"]),
        "delinquencies in last 2 years": int(row["delinq_2yrs"]),
        "credit inquiries in last 6 months": int(row["inq_last_6mths"]),
        "public records": int(row["pub_rec"]),
        "years of credit history": round(float(row["credit_history_years"]), 1),
        "term": "36 months",
    }

# One example, so we can read exactly what the AI gets
print(json.dumps(applicant_fields(base.iloc[0], "teacher"), indent=2))

{
  "job title": "teacher",
  "years employed": "2 years",
  "annual income": "$61,000",
  "home ownership": "rent",
  "loan amount": "$10,000",
  "loan purpose": "debt consolidation",
  "FICO score": 727,
  "debt-to-income ratio": "25.7%",
  "revolving utilization": "32.0%",
  "open accounts": 11,
  "total accounts": 37,
  "delinquencies in last 2 years": 0,
  "credit inquiries in last 6 months": 0,
  "public records": 0,
  "years of credit history": 12.3,
  "term": "36 months"
}


## Step 3 · Build the 560 versions

Each application × 14 title conditions. Each version gets an analysis ID like `A07_teacher`.

**The AI never sees that ID**, because it contains the grade letter and the job-title condition. Instead each version gets a random **request ID** such as `r482`, and a lookup table links the two. This keeps the experiment hidden from the AI.

In [4]:
def safe_name(title):
    return "blank" if title == "(blank)" else title.replace(" ", "_")

versions = pd.DataFrame(
    [(app, cond, f"{app}_{safe_name(cond)}") for app in base["app_id"] for cond in CONDITIONS],
    columns=["app_id", "condition", "version_id"],
)
rng = np.random.default_rng(SEED)
versions["request_id"] = [f"r{n}" for n in rng.choice(np.arange(100, 1000), size=len(versions), replace=False)]
print(f"{len(versions)} versions = {base['app_id'].nunique()} applications x {len(CONDITIONS)} conditions")
versions.head(3)

560 versions = 40 applications x 14 conditions


,app_id,condition,version_id,request_id
0,A01,software engineer,A01_software_engineer,r177
1,A01,police officer,A01_police_officer,r690
2,A01,executive assistant,A01_executive_assistant,r545


## Step 4 · Deal the versions into balanced batches

Each request holds about 25 applicants. The batches have to follow three rules:

1. **Never two versions of the same application in one batch.** Otherwise the AI would see two identical applicants with different titles side by side and could guess the experiment.
2. **6 or 7 applicants from each grade in every batch.** The AI probably judges each applicant partly against the others in its batch (it's told about 15 in 100 fail). If one batch were full of risky D loans and another full of safe A loans, where a version landed would change its score. Balanced batches make every batch look alike.
3. **At most 2 of any job title in a batch**, for the same reason.

There are 140 versions per grade spread over 23 batches, so most batches get 6 of a grade and 2 batches get 7. That makes 15 batches of 24 applicants and 8 of 25, instead of exactly 25 each: it's the only way rule 2 can hold everywhere.

**How the dealing works, like dealing cards:** take one application at a time in random order. Put each of its 14 versions into a batch that still follows all three rules, preferring batches where that grade and title are scarcest. If we ever get stuck, shuffle and start over. A different seed gives a completely different deal, which is how the repeat pass gets new batches. The code checks every rule at the end.

In [5]:
N_BATCHES = int(np.ceil(len(versions) / BATCH_SIZE))     # 560 / 25 -> 23 batches
versions["grade"] = versions["app_id"].str[0]

def deal_balanced(seed, max_tries=500):
    """Assign every version to a batch following the three rules. Returns a batch plan."""
    for attempt in range(1, max_tries + 1):
        rng = np.random.default_rng([seed, attempt])
        # How many batches may hold one MORE than the minimum, for each grade and each title
        per_grade = versions["grade"].value_counts()
        per_title = versions["condition"].value_counts()
        grade_min = {g: n // N_BATCHES for g, n in per_grade.items()}
        title_min = {t: n // N_BATCHES for t, n in per_title.items()}
        grade_extra = {g: n - grade_min[g] * N_BATCHES for g, n in per_grade.items()}
        title_extra = {t: n - title_min[t] * N_BATCHES for t, n in per_title.items()}

        size = np.zeros(N_BATCHES, int)
        g_count = {g: np.zeros(N_BATCHES, int) for g in per_grade.index}
        t_count = {t: np.zeros(N_BATCHES, int) for t in per_title.index}
        apps_in = [set() for _ in range(N_BATCHES)]
        batch_of = {}
        stuck = False
        for app in rng.permutation(versions["app_id"].unique()):
            rows = versions[versions["app_id"] == app]
            for r in rows.iloc[rng.permutation(len(rows))].itertuples():
                g, t = r.grade, r.condition
                allowed = [
                    b for b in range(N_BATCHES)
                    if app not in apps_in[b] and size[b] < BATCH_SIZE
                    and (g_count[g][b] < grade_min[g] or (g_count[g][b] == grade_min[g] and grade_extra[g] > 0))
                    and (t_count[t][b] < title_min[t] or (t_count[t][b] == title_min[t] and title_extra[t] > 0))
                ]
                if not allowed:
                    stuck = True
                    break
                b = min(allowed, key=lambda b: (g_count[g][b], t_count[t][b], size[b], rng.random()))
                if g_count[g][b] == grade_min[g]: grade_extra[g] -= 1
                if t_count[t][b] == title_min[t]: title_extra[t] -= 1
                g_count[g][b] += 1; t_count[t][b] += 1; size[b] += 1
                apps_in[b].add(app)
                batch_of[r.version_id] = b + 1
            if stuck:
                break
        if stuck:
            continue

        plan = versions.assign(batch=versions["version_id"].map(batch_of))
        plan["order_in_batch"] = plan.groupby("batch")["version_id"].transform(lambda s: rng.permutation(len(s)))
        plan = plan.sort_values(["batch", "order_in_batch"]).reset_index(drop=True)

        # Check every rule on the finished plan
        assert plan["batch"].notna().all() and len(plan) == len(versions)
        assert plan.groupby("batch")["app_id"].apply(lambda s: s.is_unique).all(), "an application repeats in a batch"
        grades = pd.crosstab(plan["grade"], plan["batch"])
        titles = pd.crosstab(plan["condition"], plan["batch"])
        assert grades.values.min() >= 6 and grades.values.max() <= 7, "grade mix out of range"
        assert titles.values.max() <= 2, "a title appears more than twice in a batch"
        assert plan.groupby("batch").size().max() <= BATCH_SIZE
        print(f"seed {seed}: valid plan found on try {attempt}. "
              f"Batch sizes {plan.groupby('batch').size().value_counts().sort_index().to_dict()}, "
              f"each grade {grades.values.min()}-{grades.values.max()} per batch, "
              f"each title {titles.values.min()}-{titles.values.max()} per batch")
        return plan
    raise RuntimeError(f"No valid plan found in {max_tries} tries")

PLAN_SEEDS = {"main_1": 2015, "main_2": 2016}     # main_2 gets a completely new deal
plans = {}
for run_name, seed in PLAN_SEEDS.items():
    plans[run_name] = deal_balanced(seed)
    path = Path(f"outputs/batch_plan_{run_name}.csv")
    if path.exists():   # a rerun must rebuild exactly the plan the saved replies were made with
        saved = pd.read_csv(path)
        assert saved["version_id"].tolist() == plans[run_name]["version_id"].tolist() \
            and saved["batch"].tolist() == plans[run_name]["batch"].astype(int).tolist(), f"{path} differs from the rebuilt plan"
    plans[run_name].to_csv(path, index=False)

same_batch = (plans["main_1"].set_index("version_id")["batch"]
              == plans["main_2"].set_index("version_id")["batch"].reindex(plans["main_1"]["version_id"]).values)
print(f"Versions that land in the same batch number in both plans: {same_batch.mean():.0%} (chance level is about 4%)")

seed 2015: valid plan found on try 8. Batch sizes {24: 15, 25: 8}, each grade 6-7 per batch, each title 1-2 per batch


seed 2016: valid plan found on try 9. Batch sizes {24: 15, 25: 8}, each grade 6-7 per batch, each title 1-2 per batch
Versions that land in the same batch number in both plans: 4% (chance level is about 4%)


## Step 5 · Ask Gemini, safely

The prompt comes from `prompts/plain.txt`, so anyone can read exactly what we asked. The applicants are pasted in where it says `{applications}`.

The function below handles one batch and protects long runs:

- **Every reply is saved** as its own file in `outputs/gemini_raw/<run_name>/`. The file also holds the model name, the model version Google reports, the prompt file, the full prompt sent, the batch number, the time, how long the request took, and how many API calls it needed.
- **Reruns skip finished batches.** If a run stops halfway, running it again picks up where it left off.
- **Busy or rate-limit errors** (429, 500, 503, 504), **dropped connections and timeouts** are retried up to 5 tries, waiting 5, 10, 20, then 40 seconds. If all 5 fail, the run **stops** rather than hammering the API.
- **Every answer is checked:** the IDs must match the batch exactly (none missing, none extra, none repeated), and every probability must be a number from 0 to 100. A bad reply gets one retry, then the batch is logged as failed and the run moves on.
- **A 13-second pause between requests** keeps us under 5 requests a minute.

In [6]:
BASE_BY_APP = base.set_index("app_id")
RETRYABLE = {429, 500, 503, 504}
WAITS = [5, 10, 20, 40]           # seconds between the 5 tries
PAUSE_BETWEEN_REQUESTS = 13       # stays under 5 requests a minute
_last_request = [0.0]

def wait_for_turn(pause):
    """Sleeps until at least `pause` seconds have passed since the last request."""
    wait = pause - (time.monotonic() - _last_request[0])
    if wait > 0:
        time.sleep(wait)
    _last_request[0] = time.monotonic()

def build_prompt(batch_rows, prompt_file):
    template = Path(prompt_file).read_text(encoding="utf-8")
    lines = [json.dumps({"id": r.request_id, **applicant_fields(BASE_BY_APP.loc[r.app_id], r.condition)})
             for r in batch_rows.itertuples()]
    return template.replace("{applications}", "\n".join(lines))

def check_reply(text, expected_ids):
    """Returns (scores, problems). scores maps request ID -> probability; problems is a list of strings."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        return {}, ["no JSON list found"]
    try:
        items = json.loads(text[start : end + 1])
    except json.JSONDecodeError as e:
        return {}, [f"JSON could not be read: {e}"]
    problems, scores = [], {}
    for item in items:
        if not isinstance(item, dict) or "id" not in item or "probability" not in item:
            problems.append(f"badly formed item: {item!r}"); continue
        rid, p = str(item["id"]), item["probability"]
        if rid in scores:
            problems.append(f"repeated id {rid}")
        if isinstance(p, bool) or not isinstance(p, (int, float)) or not 0 <= p <= 100:
            problems.append(f"bad probability for {rid}: {p!r}"); continue
        scores[rid] = float(p)
    missing, extra = set(expected_ids) - set(scores), set(scores) - set(expected_ids)
    if missing: problems.append(f"missing ids: {sorted(missing)}")
    if extra: problems.append(f"extra ids: {sorted(extra)}")
    return scores, problems

def call_model(prompt, model, pause, waits=WAITS):
    """One request, retrying busy/rate-limit errors. Returns (response or None, seconds, error_log, fatal_message)."""
    error_log = []
    for attempt in range(1, 6):
        wait_for_turn(pause)
        started = time.perf_counter()
        try:
            response = client.models.generate_content(model=model, contents=prompt)
            return response, time.perf_counter() - started, error_log, None
        except errors.APIError as e:
            error_log.append({"try": attempt, "code": e.code, "status": e.status, "message": e.message,
                              "seconds": round(time.perf_counter() - started, 2)})
            if e.code not in RETRYABLE or attempt == 5:
                return None, None, error_log, f"API error {e.code} {e.status}: {e.message}"
            time.sleep(waits[attempt - 1])
        except NETWORK_ERRORS as e:   # the server dropped the connection, or the 4-minute limit ran out
            error_log.append({"try": attempt, "code": "network", "status": type(e).__name__, "message": str(e)[:300],
                              "seconds": round(time.perf_counter() - started, 2)})
            if attempt == 5:
                return None, None, error_log, f"Network error {type(e).__name__}: {e}"
            time.sleep(waits[attempt - 1])

def run_batch(run_name, plan, batch_number, model=MODEL, prompt_file=PROMPT_FILE, pause=PAUSE_BETWEEN_REQUESTS,
              waits=WAITS):
    """Runs one batch and saves its record. Returns the batch's status."""
    out_dir = Path("outputs/gemini_raw") / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"batch_{batch_number:02d}.json"
    earlier = []
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        if old["status"] == "ok":
            print(f"{run_name} batch {batch_number:02d}: already saved, skipping")
            return "ok"
        # A failed batch is tried again, but its earlier failed attempts are kept in the record
        earlier = old.get("earlier_failed_attempts", []) + old["attempts"]

    rows = plan[plan["batch"] == batch_number]
    prompt = build_prompt(rows, prompt_file)
    expected = rows["request_id"].tolist()
    record = {"run_name": run_name, "model": model, "prompt_file": str(prompt_file), "batch": batch_number,
              "request_ids": expected, "prompt_sent": prompt, "attempts": [], "earlier_failed_attempts": earlier}

    for validity_try in (1, 2):        # one retry if the reply fails the checks
        stamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
        response, seconds, error_log, fatal = call_model(prompt, model, pause, waits)
        if fatal:
            record["attempts"].append({"timestamp_utc": stamp, "api_calls": len(error_log),
                                       "api_errors": error_log, "error": fatal})
            record["status"] = "failed"
            path.write_text(json.dumps(record, indent=2), encoding="utf-8")
            raise RuntimeError(f"{run_name} batch {batch_number}: {fatal}. Run stopped.")
        text = response.text or ""
        scores, problems = check_reply(text, expected)
        usage = response.usage_metadata
        record["attempts"].append({
            "timestamp_utc": stamp, "seconds": round(seconds, 2), "model_version": response.model_version,
            "api_calls": len(error_log) + 1, "api_errors": error_log, "raw_reply": text, "problems": problems,
            "prompt_tokens": getattr(usage, "prompt_token_count", None),
            "reply_tokens": getattr(usage, "candidates_token_count", None),
        })
        if not problems:
            record["status"], record["scores"] = "ok", scores
            break
    else:
        record["status"] = "failed"
    path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    last = record["attempts"][-1]
    print(f"{run_name} batch {batch_number:02d}: {record['status']}, {len(record['attempts'])} attempt(s), "
          f"{last['seconds']}s, problems: {last['problems'] or 'none'}")
    return record["status"]

def run(run_name, plan, batches, model=MODEL, prompt_file=PROMPT_FILE, pause=PAUSE_BETWEEN_REQUESTS,
        waits=WAITS, stop_on_failure=False):
    for b in batches:
        status = run_batch(run_name, plan, int(b), model, prompt_file, pause, waits)   # plain int: numpy ints can't be saved as JSON
        if stop_on_failure and status != "ok":
            raise RuntimeError(f"{run_name} batch {int(b)} failed its checks. Run stopped, as instructed.")

def load_run(run_name):
    return [json.loads(p.read_text(encoding="utf-8"))
            for p in sorted((Path("outputs/gemini_raw") / run_name).glob("batch_*.json"))]

def run_summary(run_name):
    records = load_run(run_name)
    return {
        "run": run_name,
        "batches_ok": sum(r["status"] == "ok" for r in records),
        "batches_failed": sum(r["status"] != "ok" for r in records),
        "valid_answers": sum(len(r.get("scores", {})) for r in records),
        "api_requests": sum(a.get("api_calls", 1) for r in records for a in r["attempts"] + r.get("earlier_failed_attempts", [])),
        "api_errors": sum(len(a.get("api_errors", a.get("api_errors_before_success", [])))
                          for r in records for a in r["attempts"] + r.get("earlier_failed_attempts", [])),
        "reply_retries": sum(len(r["attempts"]) - 1 for r in records),
        "median_seconds": np.median([a["seconds"] for r in records for a in r["attempts"] if a.get("seconds")]),
        "model_versions": ", ".join(sorted({a.get("model_version") or "?" for r in records for a in r["attempts"] if "raw_reply" in a})),
    }

## Step 6 · Pilot 2: check the balanced batches work

Runs the first 3 batches of the `main_1` plan under the run name `pilot_2`, in its own folder.

**Both pilots are kept out of every analysis.** `pilot` used the first, unbalanced batch plan (saved as `outputs/batch_plan_pilot.csv` for the record). `pilot_2` is a final plumbing check. Only `main_1` and `main_2` are ever analysed.

**Go/no-go:** the main runs start only if every ID in `pilot_2` comes back valid.

In [7]:
run("pilot_2", plans["main_1"], [1, 2, 3])

pilot_records = load_run("pilot_2")
expected_answers = sum(len(r["request_ids"]) for r in pilot_records)
GO = len(pilot_records) == 3 and all(r["status"] == "ok" for r in pilot_records)
print(f"\npilot_2: {sum(len(r.get('scores', {})) for r in pilot_records)} of {expected_answers} IDs valid  ->  {'GO' if GO else 'NO-GO: main runs will be skipped'}")

pilot_2 batch 01: already saved, skipping
pilot_2 batch 02: already saved, skipping
pilot_2 batch 03: already saved, skipping

pilot_2: 74 of 74 IDs valid  ->  GO


## Step 7 · Main pass and repeat pass

- **`main_1`:** all 560 versions, using the `main_1` balanced plan (23 requests).
- **`main_2`:** the same 560 versions, re-dealt into new balanced batches (the `main_2` plan, a different seed), in a different order.

Nothing about the versions changes between the two runs; only the batches they sit in and the AI's own randomness. So the difference between `main_1` and `main_2` for the same version is our **noise floor**: it includes the AI's run-to-run wobble and any effect of which other applicants share the batch. A job-title effect smaller than this noise can't be claimed.

Each run takes about 23 × 15 seconds ≈ 6 minutes.

In [8]:
if GO:
    for run_name in ["main_1", "main_2"]:
        plan = plans[run_name]
        run(run_name, plan, sorted(plan["batch"].unique()))
        print()
else:
    print("Skipped: pilot_2 did not pass.")

main_1 batch 01: already saved, skipping
main_1 batch 02: already saved, skipping
main_1 batch 03: already saved, skipping
main_1 batch 04: already saved, skipping
main_1 batch 05: already saved, skipping
main_1 batch 06: already saved, skipping
main_1 batch 07: already saved, skipping
main_1 batch 08: already saved, skipping
main_1 batch 09: already saved, skipping
main_1 batch 10: already saved, skipping
main_1 batch 11: already saved, skipping
main_1 batch 12: already saved, skipping
main_1 batch 13: already saved, skipping
main_1 batch 14: already saved, skipping
main_1 batch 15: already saved, skipping
main_1 batch 16: already saved, skipping
main_1 batch 17: already saved, skipping
main_1 batch 18: already saved, skipping
main_1 batch 19: already saved, skipping
main_1 batch 20: already saved, skipping
main_1 batch 21: already saved, skipping
main_1 batch 22: already saved, skipping
main_1 batch 23: already saved, skipping

main_2 batch 01: already saved, skipping
main_2 batch 02

## Noise floor: main_1 vs main_2

For every version scored in both runs, the difference between its two scores. The **average absolute difference** is how far the AI's answer moves for the exact same applicant, same title, when only its batch and the AI's randomness change.

This is not yet an analysis of job titles. That comes next, and it will use this number as the bar to clear.

The scores from both runs are saved to `outputs/ai_scores.csv` for the analysis notebook.

In [9]:
def scores_for(run_name):
    return pd.Series({rid: p for r in load_run(run_name) for rid, p in r.get("scores", {}).items()}, name=run_name)

if GO:
    both = versions[["version_id", "app_id", "condition", "request_id"]].copy()
    for run_name in ["main_1", "main_2"]:
        both[run_name] = both["request_id"].map(scores_for(run_name))
        both[f"batch_{run_name}"] = both["version_id"].map(plans[run_name].set_index("version_id")["batch"])
    both.drop(columns="request_id").to_csv("outputs/ai_scores.csv", index=False)

    paired = both.dropna(subset=["main_1", "main_2"])
    diff = paired["main_2"] - paired["main_1"]
    rng = np.random.default_rng(SEED)
    boot = [diff.abs().to_numpy()[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2_000)]
    print(f"Versions scored in both runs: {len(paired)} of {len(versions)}")
    print(f"Average absolute difference: {diff.abs().mean():.2f} points "
          f"(95% interval {np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f})")
    print(f"Median absolute difference:  {diff.abs().median():.2f} points")
    print(f"Identical in both runs:      {(diff == 0).mean():.0%} of versions")
    print(f"Largest difference:          {diff.abs().max():.1f} points")
    print(f"Average shift main_2 - main_1: {diff.mean():+.2f} points (near 0 means neither run is higher overall)")
    print(f"Correlation between runs:    {paired['main_1'].corr(paired['main_2']):.3f}")
    print(f"\nAverage score: main_1 {paired['main_1'].mean():.1f}, main_2 {paired['main_2'].mean():.1f}")

Versions scored in both runs: 560 of 560
Average absolute difference: 1.97 points (95% interval 1.82 to 2.11)
Median absolute difference:  1.50 points
Identical in both runs:      13% of versions
Largest difference:          8.0 points
Average shift main_2 - main_1: +0.32 points (near 0 means neither run is higher overall)
Correlation between runs:    0.867

Average score: main_1 16.3, main_2 16.6


## Step 8 · Robustness runs: does the result depend on the wording or the model?

`PREREGISTRATION.md` plans robustness checks with other prompt wordings and a second model. Both runs below use **the same 560 versions and the same batch plan as `main_1`**, so the only thing that changes is the wording or the model.

- **`persona`:** the same model (`gemini-3.5-flash-lite`) with `prompts/loan_officer.txt`. It is the plain prompt plus one opening line: *"You are an experienced loan officer at a lending company reviewing personal loan applications."* Role-play wording like this is a common way people use AI in real lending tools, and it might make the AI lean on job titles more.
- **`gemma`:** the plain prompt on `gemma-4-31b-it`, an open model from Google's separate Gemma family, with default settings. Gemma gets a longer pause between requests (25 seconds) to stay well inside its smaller per-minute token limit. If any Gemma batch fails, the run **stops** rather than switching to another model.

The first Gemma attempt stopped at batch 5 when Google's servers returned "internal error" and "high demand" five times in a row. Gemma now waits longer between retries (30 seconds, 1, 2, then 3 minutes). Its first 4 batches were already saved and are skipped; batch 5's failed attempts stay in its file for the record.

The second attempt reached batch 9, where one request hung for about 10 minutes and then the server dropped the connection without replying. Network errors like that weren't being retried, so the run stopped. Now dropped connections and timeouts are retried like busy errors, and every request has a 4-minute limit. Batches 1–8 are skipped. That lost batch-9 request left no file, because it crashed before saving.

**Outcome: Gemma was dropped.** On the third attempt, batch 9 succeeded, but batch 10 failed all 5 tries with "internal error" (500) from Google's servers. We had decided in advance that one more full failure meant dropping the second-model check, so it stopped there, with 9 of 23 batches done. A partial run covers only some versions of each application, so its title effects would be unbalanced and are **not analysed**. The saved Gemma replies stay in `outputs/gemini_raw/gemma/` as a record. The cell below now runs only `persona` (already saved, so it is skipped).

In [10]:
ROBUSTNESS_RUNS = {
    "persona": {"model": "gemini-3.5-flash-lite", "prompt_file": Path("prompts/loan_officer.txt"), "pause": 13},
    "gemma":   {"model": "gemma-4-31b-it",        "prompt_file": Path("prompts/plain.txt"),        "pause": 25,
                "waits": [30, 60, 120, 180]},
}

In [11]:
DROPPED = {"gemma": "stopped at batch 10 after 5 failed tries (server error 500); dropped as decided in advance"}
plan = plans["main_1"]
for run_name, settings in ROBUSTNESS_RUNS.items():
    if run_name in DROPPED:
        print(f"{run_name}: not run again ({DROPPED[run_name]})")
        continue
    run(run_name, plan, sorted(plan["batch"].unique()), stop_on_failure=True, **settings)
    print()

persona batch 01: already saved, skipping
persona batch 02: already saved, skipping
persona batch 03: already saved, skipping
persona batch 04: already saved, skipping
persona batch 05: already saved, skipping
persona batch 06: already saved, skipping
persona batch 07: already saved, skipping
persona batch 08: already saved, skipping
persona batch 09: already saved, skipping
persona batch 10: already saved, skipping
persona batch 11: already saved, skipping
persona batch 12: already saved, skipping
persona batch 13: already saved, skipping
persona batch 14: already saved, skipping
persona batch 15: already saved, skipping
persona batch 16: already saved, skipping
persona batch 17: already saved, skipping
persona batch 18: already saved, skipping
persona batch 19: already saved, skipping
persona batch 20: already saved, skipping
persona batch 21: already saved, skipping
persona batch 22: already saved, skipping
persona batch 23: already saved, skipping

gemma: not run again (stopped at 

## Save the robustness scores

One row per version with the `persona` and `gemma` scores, saved to `outputs/ai_scores_robustness.csv` for notebook 05.

In [12]:
robust = versions[["version_id", "app_id", "condition", "request_id"]].copy()
for run_name in ROBUSTNESS_RUNS:
    robust[run_name] = robust["request_id"].map(scores_for(run_name))
robust.drop(columns="request_id").to_csv("outputs/ai_scores_robustness.csv", index=False)
print(robust[list(ROBUSTNESS_RUNS)].notna().sum().rename("versions scored").to_string())
robust[list(ROBUSTNESS_RUNS)].describe().round(1)

persona    560
gemma      220


,persona,gemma
count,560.0,220.0
mean,16.6,18.2
std,5.0,6.8
min,3.0,2.0
25%,13.0,14.8
50%,17.0,18.0
75%,20.0,22.0
max,29.0,38.0


## Step 9 · Competence check: can the AI rank risk at all?

A pre-registered control. Before trusting what an AI loan officer does with job titles, check that it can do the basic job: tell riskier loans from safer ones.

- **500 random real 2015 loans** with every field filled in, any grade, any income, not including the 40 base applications.
- **Job title left blank** for all of them, so this measures the financial fields alone.
- Same model (`gemini-3.5-flash-lite`), same plain prompt, same fields, same safety and resume code, batches of 25 (20 requests), run name `competence`.

Then we compare **AUC** on exactly these 500 loans: Gemini's probability vs LendingClub's sub-grade (and, for reference, our model and the interest rate). AUC is the chance that a randomly picked loan that defaulted is rated riskier than a randomly picked one that was repaid: 0.5 is a coin flip, 1.0 is perfect. With about 75 defaults among 500 loans, the intervals will be fairly wide.

In [13]:
COMPETENCE_N = 500
SUBGRADE_ORDER = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]

eligible = loans.dropna(subset=AI_FIELDS)
eligible = eligible[~eligible["id"].isin(base["id"])]
comp = eligible.sample(COMPETENCE_N, random_state=SEED).reset_index(drop=True)
comp.insert(0, "app_id", [f"K{i:03d}" for i in range(1, COMPETENCE_N + 1)])
BASE_BY_APP = pd.concat([base, comp]).set_index("app_id")   # build_prompt looks applicants up here

rng = np.random.default_rng(SEED + 1)
comp_plan = pd.DataFrame({
    "app_id": comp["app_id"], "condition": "(blank)",
    "request_id": [f"c{n}" for n in rng.choice(np.arange(100, 1000), size=COMPETENCE_N, replace=False)],
})
comp_plan = comp_plan.iloc[rng.permutation(COMPETENCE_N)].reset_index(drop=True)
comp_plan["batch"] = comp_plan.index // BATCH_SIZE + 1
comp_plan.to_csv("outputs/batch_plan_competence.csv", index=False)

print(f"{COMPETENCE_N} loans in {comp_plan['batch'].nunique()} batches; {comp['default'].sum()} defaulted "
      f"({comp['default'].mean():.1%})")
print("Grades:", comp["grade"].value_counts().sort_index().to_dict())

500 loans in 20 batches; 68 defaulted (13.6%)


Grades: {'A': 124, 'B': 172, 'C': 131, 'D': 57, 'E': 14, 'F': 1, 'G': 1}


In [14]:
run("competence", comp_plan, sorted(comp_plan["batch"].unique()))

competence batch 01: already saved, skipping
competence batch 02: already saved, skipping
competence batch 03: already saved, skipping
competence batch 04: already saved, skipping
competence batch 05: already saved, skipping
competence batch 06: already saved, skipping
competence batch 07: already saved, skipping
competence batch 08: already saved, skipping
competence batch 09: already saved, skipping
competence batch 10: already saved, skipping
competence batch 11: already saved, skipping
competence batch 12: already saved, skipping
competence batch 13: already saved, skipping
competence batch 14: already saved, skipping
competence batch 15: already saved, skipping
competence batch 16: already saved, skipping
competence batch 17: already saved, skipping
competence batch 18: already saved, skipping
competence batch 19: already saved, skipping
competence batch 20: already saved, skipping


## Competence result

All AUCs are on the same loans, and the intervals come from a **paired bootstrap**: resample the 500 loans 2,000 times, and recompute every AUC on each resample. Pairing matters for the difference between Gemini and the sub-grade. Both are scored on the same resampled loans, so the interval reflects only how much they really differ.

In [15]:
from sklearn.metrics import roc_auc_score

comp["ai_probability"] = comp["app_id"].map(comp_plan.set_index("app_id")["request_id"]).map(scores_for("competence"))
comp["subgrade_rank"] = comp["sub_grade"].map({s: i for i, s in enumerate(SUBGRADE_ORDER)})
scored = comp.dropna(subset=["ai_probability"]).reset_index(drop=True)

RANKERS = {"Gemini (plain prompt, no job title)": "ai_probability", "LendingClub sub-grade": "subgrade_rank",
           "LendingClub interest rate": "int_rate", "Our model (notebook 02)": "model_risk"}
y = scored["default"].to_numpy()
point = {name: roc_auc_score(y, scored[col]) for name, col in RANKERS.items()}

rng = np.random.default_rng(SEED)
boot = {name: [] for name in RANKERS}
boot_diff = []
for _ in range(2_000):
    i = rng.integers(0, len(scored), len(scored))
    if y[i].min() == y[i].max():
        continue
    aucs = {name: roc_auc_score(y[i], scored[col].to_numpy()[i]) for name, col in RANKERS.items()}
    for name in RANKERS:
        boot[name].append(aucs[name])
    boot_diff.append(aucs["Gemini (plain prompt, no job title)"] - aucs["LendingClub sub-grade"])

competence = pd.DataFrame({
    "auc": point,
    "low": {n: np.percentile(b, 2.5) for n, b in boot.items()},
    "high": {n: np.percentile(b, 97.5) for n, b in boot.items()},
})
competence["loans"], competence["defaults"] = len(scored), int(y.sum())
competence.round(4).to_csv("outputs/competence_auc.csv")
scored[["app_id", "id", "grade", "sub_grade", "int_rate", "default", "model_risk", "ai_probability"]].to_csv(
    "outputs/competence_scores.csv", index=False)

d_lo, d_hi = np.percentile(boot_diff, [2.5, 97.5])
print(f"Scored {len(scored)} of {COMPETENCE_N} loans ({int(y.sum())} defaults)")
print(f"Gemini minus sub-grade AUC: {point['Gemini (plain prompt, no job title)'] - point['LendingClub sub-grade']:+.3f} "
      f"(95% interval {d_lo:+.3f} to {d_hi:+.3f})")
competence.round(3)

Scored 500 of 500 loans (68 defaults)
Gemini minus sub-grade AUC: -0.032 (95% interval -0.105 to +0.035)


,auc,low,high,loans,defaults
"Gemini (plain prompt, no job title)",0.625,0.553,0.696,500,68
LendingClub sub-grade,0.657,0.589,0.728,500,68
LendingClub interest rate,0.657,0.589,0.728,500,68
Our model (notebook 02),0.684,0.608,0.754,500,68


## Run report

Reads every saved file back from disk, the same way the analysis will. For each run it counts valid answers, failed batches, and every API request actually sent, including retries.

**Not in these files:**
- one `main_1` request whose reply was lost to a saving bug, then re-asked (fixed in this notebook);
- one `gemma` batch-9 request that crashed on a dropped connection before it could be saved;
- the one-off connection tests in `test_gemini.py`.

In [16]:
RUNS = ["pilot", "pilot_2", "main_1", "main_2", "persona", "gemma", "competence"]
report = pd.DataFrame([run_summary(r) for r in RUNS if (Path("outputs/gemini_raw") / r).exists()]).set_index("run")
report.loc["total"] = report[["batches_ok", "batches_failed", "valid_answers", "api_requests", "api_errors", "reply_retries"]].sum()
report

,batches_ok,batches_failed,valid_answers,api_requests,api_errors,reply_retries,median_seconds,model_versions
run,,,,,,,,
pilot,3,0,75,3,0,0,2.93,gemini-3.5-flash-lite
pilot_2,3,0,74,3,0,0,1.89,gemini-3.5-flash-lite
main_1,23,0,560,29,6,0,2.28,gemini-3.5-flash-lite
main_2,23,0,560,33,10,0,1.84,gemini-3.5-flash-lite
persona,23,0,560,23,0,0,1.64,gemini-3.5-flash-lite
gemma,9,1,220,29,20,0,98.01,gemma-4-31b-it
competence,20,0,500,20,0,0,1.65,gemini-3.5-flash-lite
total,104,1,2549,140,36,0,NaN,NaN
